# BME 603/606: Computational Methods for Biological Modeling & Simulation
## In-Class Activity: Sequence Alignment

Today's topic: **Sequence Alignment**

1. **FASTA format** — learn what a FASTA file is, download real biological sequences, and implement a parser.

2. **Global alignment (Needleman–Wunsch)** — fill the DP table and trace back to recover the optimal end-to-end alignment.

3. **Local alignment (Smith–Waterman)** — find the best matching sub-region between a large nucleotide sequence and a smaller query.

4. **Apply to real sequences** — align hemoglobin protein sequences from multiple species.

5. **Challenge (bonus)** — re-implement global alignment using the **BLOSUM62** substitution matrix and align multiple species.

Let's get started!

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time
import os
import urllib.request

# Plotting defaults
plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

print("Imports done!")

---
## Part 1: Working with Real Biological Sequences

### Background: The FASTA Format

**FASTA** is the standard text-based format for representing nucleotide or amino acid sequences. It originated from the FASTA software package and is now universal in bioinformatics.

A FASTA file contains one or more sequence records. Each record has:
- A **header line** starting with `>` (greater-than), followed by a description or identifier
- One or more **sequence lines** immediately after, containing the letters (A, C, G, T for DNA; the 20 amino acid codes for proteins)

**Example single-sequence FASTA:**
```
>sp|P69905|HBA_HUMAN Hemoglobin subunit alpha OS=Homo sapiens OX=9606 GN=HBA1 PE=1 SV=2
MVLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSHGSAQVKGHGKKVADALTNAVAHVDDMPNALSALSDLHAHKLRVDPVNFKLLSHCLLVTLAAHLPAEFTPAVHASLDKFLASVSTVLTSKYR
```

**Key points:**
- The header line can be any length and contains metadata (identifier, organism, gene name, etc.)
- Sequence lines are typically wrapped at 60–80 characters
- Nucleotide codes: A, C, G, T, U, N (any), and ambiguity codes like R (purine), Y (pyrimidine), etc.

### Where to Download Sequences

| Database | What it contains | URL |
|---|---|---|
| **NCBI Datasets** | Genes, genomes, proteins | https://www.ncbi.nlm.nih.gov/datasets/ |
| **UniProt** | Protein sequences | https://www.uniprot.org/ |
| **Ensembl FTP** | Genome and protein databases | https://ftp.ensembl.org/ |

**For this exercise**, we'll download sequences from UniProt using their REST API. The URL format is:
```
https://rest.uniprot.org/uniprotkb/{ACCESSION}.fasta
```

### Exercise 1: Parse a FASTA File

**Tasks:**
1. Download the FASTA files for several hemoglobin subunit alpha (HBA1) proteins from UniProt.
2. Implement `read_fasta(filepath)` that parses a FASTA file and returns a dictionary mapping sequence identifier → sequence string.
3. Verify by printing the identifiers and sequence lengths.

**Download links (UniProt):**
- Human HBA1: `https://rest.uniprot.org/uniprotkb/P69905.fasta`
- Mouse HBA1: `https://rest.uniprot.org/uniprotkb/P01942.fasta`
- Bovine HBA1: `https://rest.uniprot.org/uniprotkb/P01966.fasta`

**Hint:** The sequence lines may be wrapped; you need to concatenate them after skipping the header line.

In [ ]:
def read_fasta(filepath):
    """
    Parse a FASTA file and return a dictionary of {identifier: sequence}.

    Parameters
    ----------
    filepath : str — path to a FASTA file

    Returns
    -------
    dict — mapping from sequence ID (first word after '>') to sequence string
    """
    sequences = {}
    current_id = None
    current_seq = []
    # Your code here — open the file, skip '>' lines, concatenate sequence lines
    return sequences


# Download helper
def download_fasta(url, filepath):
    """Download a FASTA file if it doesn't already exist."""
    if not os.path.exists(filepath):
        try:
            urllib.request.urlretrieve(url, filepath)
            print(f"Downloaded {filepath}")
        except Exception as e:
            print(f"Download failed for {url}: {e}")
    else:
        print(f"Already have {filepath}")


# Download the three hemoglobin sequences
download_fasta("https://rest.uniprot.org/uniprotkb/P69905.fasta", "HBA_HUMAN.fasta")
download_fasta("https://rest.uniprot.org/uniprotkb/P01942.fasta", "HBA_MOUSE.fasta")
download_fasta("https://rest.uniprot.org/uniprotkb/P01966.fasta", "HBA_BOVIN.fasta")

# Your code here — parse each file and print the identifiers and lengths

---
## Part 2: Global Sequence Alignment (Needleman–Wunsch)

### Background: Global Alignment

**Global alignment** aligns two sequences end-to-end, from the first character to the last. It is optimal when the sequences are similar in length and expected to be homologous across their full length.

The **Needleman–Wunsch algorithm** (1970) uses dynamic programming to find the highest-scoring alignment. The recurrence is:

$$dp[i][j] = \max \begin{cases} dp[i-1][j-1] + s(A[i-1], B[j-1]) & \text{(diagonal: match/mismatch)} \\ dp[i-1][j] + g & \text{(up: gap in B)} \\ dp[i][j-1] + g & \text{(left: gap in A)} \end{cases}$$

where $s(a,b)$ is the substitution score and $g$ is the gap penalty.

**Base cases:**
- $dp[0][0] = 0$
- $dp[i][0] = i \times g$
- $dp[0][j] = j \times g$

### Exercise 2: Implement Global Alignment

**Tasks:**
1. Implement `global_align(A, B, match_score, mismatch_score, gap_penalty)` returning both the score and the filled DP table.
2. Implement `global_traceback(dp, A, B, match_score, mismatch_score, gap_penalty)` returning the aligned sequences.
3. Test on the lecture example: `A = "TGCAGTAGGAT"`, `B = "TCCAATAGAT"` with match=+1, mismatch=0, gap=-1. Verify the score is 7.

**Hint:** The traceback starts at `(len(A), len(B))` and walks backward to `(0,0)`, collecting gap characters (`-`) when moving vertically or horizontally.

In [ ]:
def global_align(A, B, match_score=1, mismatch_score=0, gap_penalty=-1):
    """
    Needleman-Wunsch global alignment.

    Parameters
    ----------
    A : str — first sequence
    B : str — second sequence
    match_score : int or float
    mismatch_score : int or float
    gap_penalty : int or float

    Returns
    -------
    score : int or float — optimal alignment score
    dp : 2-D list — filled DP table, shape (len(A)+1, len(B)+1)
    """
    m, n = len(A), len(B)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    # Your code here — initialize first row and column, then fill the table
    return dp[m][n], dp


def global_traceback(dp, A, B, match_score=1, mismatch_score=0, gap_penalty=-1):
    """
    Recover the optimal global alignment from a filled DP table.

    Returns
    -------
    aligned_A : str — A with gaps inserted
    aligned_B : str — B with gaps inserted
    """
    i, j = len(A), len(B)
    aligned_A, aligned_B = [], []
    # Your code here — walk backward, decide direction, prepend characters
    return "".join(reversed(aligned_A)), "".join(reversed(aligned_B))


# Your code here — run on the lecture example and verify the score

### Exercise 3: Global Alignment of Multiple Species

Now align the hemoglobin sequences you downloaded.

**Tasks:**
1. Align human HBA1 vs mouse HBA1.
2. Align human HBA1 vs bovine HBA1.
3. Align mouse HBA1 vs bovine HBA1.
4. Print the alignment score and the aligned sequences for each pair.
5. Which pair is most similar? Least similar?

In [ ]:
# Your code here — load parsed sequences, run global alignment for each pair,
# print scores and alignments

---
## Part 3: Local Sequence Alignment (Smith–Waterman)

### Background: Local Alignment

**Local alignment** finds the best matching sub-regions within two sequences, ignoring the rest. It is essential when sequences may share a conserved domain but differ elsewhere (e.g., a gene inside a large genomic region).

The **Smith–Waterman algorithm** (1981) modifies Needleman–Wunsch with three changes:
1. The score at any cell is never negative — it is **clamped to 0**.
2. The traceback starts at the **highest-scoring cell** anywhere in the table.
3. Traceback **stops when a 0 is reached**.

**Recurrence:**
$$dp[i][j] = \max \begin{cases} 0 & \text{(start fresh)} \\ dp[i-1][j-1] + s(A[i-1], B[j-1]) \\ dp[i-1][j] + g \\ dp[i][j-1] + g \end{cases}$$

### Exercise 4: Local Alignment of a Query Against a Larger Sequence

For local alignment, we will work with **nucleotide** sequences: a larger parent sequence and a smaller query sequence.

**Tasks:**
1. Download a large nucleotide sequence (e.g., a human mRNA or gene) from NCBI.
2. Download a smaller query sequence (e.g., a conserved exon or a short gene fragment).
3. Implement `local_align(A, B, match_score, mismatch_score, gap_penalty)` returning the best score and the DP table.
4. Implement `local_traceback(dp, A, B, ...)` returning the aligned local sub-sequences and the starting positions.
5. Run local alignment of the query against the parent. Where does the query align?

**Download links (NCBI Nucleotide — use E-utilities):**

You can fetch FASTA sequences from NCBI using the E-utilities API:
```
https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi?db=nucleotide&id={ACCESSION}&rettype=fasta&retmode=text
```

**Suggested sequences:**
- **Parent (large):** Human HBB mRNA (accession NM_000518)
- **Query (small):** A conserved fragment from the HBB coding region (you can extract a 100–200 bp fragment from the parent yourself)

**Alternative:** Use the human HBA1 mRNA as the parent and a short exon fragment as the query.

**Hint:** For traceback, first find the maximum value in the table. Then walk backward until you hit a 0.

In [ ]:
def local_align(A, B, match_score=1, mismatch_score=0, gap_penalty=-1):
    """
    Smith-Waterman local alignment.

    Returns
    -------
    best_score : int or float — highest score in the table
    dp : 2-D list — filled DP table
    """
    m, n = len(A), len(B)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    # Your code here — fill table with max(0, ...) at each cell
    best_score = max(max(row) for row in dp)
    return best_score, dp


def local_traceback(dp, A, B, match_score=1, mismatch_score=0, gap_penalty=-1):
    """
    Recover the optimal local alignment from a filled DP table.

    Returns
    -------
    aligned_A : str — local A segment with gaps
    aligned_B : str — local B segment with gaps
    start_A : int — start position in A
    start_B : int — start position in B
    score : int or float — alignment score
    """
    m, n = len(A), len(B)
    best_i, best_j = 0, 0
    # Your code here — find the maximum cell
    i, j = best_i, best_j
    aligned_A, aligned_B = [], []
    # Your code here — walk backward until dp[i][j] == 0
    return "".join(reversed(aligned_A)), "".join(reversed(aligned_B)), None, None, dp[best_i][best_j]


# Download a large nucleotide sequence (human HBB mRNA)
hbb_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi?db=nucleotide&id=NM_000518&rettype=fasta&retmode=text"
hbb_path = "HBB_mRNA.fasta"

if not os.path.exists(hbb_path):
    try:
        urllib.request.urlretrieve(hbb_url, hbb_path)
        print("Downloaded HBB_mRNA.fasta")
    except Exception as e:
        print(f"Download failed: {e}")

# Your code here — parse the HBB mRNA, extract a short query fragment,
# run local alignment, and print where it aligns

---
## Part 4: Applying Alignment to Real Protein Sequences

### Exercise 5: Compare Global vs. Local Alignment on Proteins

**Tasks:**
1. Run global alignment on human HBA1 vs mouse HBA1.
2. Run local alignment on the same pair.
3. Run global alignment on human HBA1 vs bovine HBA1.
4. Run local alignment on the same pair.
5. Compare the scores: When does local alignment give a higher score than global? Why?

In [ ]:
# Your code here — run both global and local alignment on the protein pairs
# Print scores and aligned sequences

---
## Part 5: Challenge Exercise (Bonus)

### Exercise 6: Global Alignment with BLOSUM62

**For bonus credit.** Simple match/mismatch scoring treats all amino acid substitutions equally. But in reality, some substitutions are more conservative than others (e.g., isoleucine ↔ valine is far less disruptive than cysteine ↔ tryptophan). The **BLOSUM62** matrix captures this by assigning scores based on observed substitution frequencies in conserved protein families.

BLOSUM62 is a 20×20 matrix where rows and columns correspond to the 20 standard amino acids (in alphabetical order: A, R, N, D, C, Q, E, G, H, I, L, K, M, F, P, S, T, W, Y, V). The score for substituting amino acid `a` with amino acid `b` is `BLOSUM62[a][b]`.

**BLOSUM62 Matrix:**

```python
# BLOSUM62 substitution matrix
# Amino acid order: A R N D C Q E G H I L K M F P S T W Y V
BLOSUM62 = {
    'A': {'A':  4, 'R': -1, 'N': -2, 'D': -2, 'C':  0, 'Q': -1, 'E': -1, 'G':  0, 'H': -2, 'I': -1, 'L': -1, 'K': -1, 'M': -1, 'F': -2, 'P': -1, 'S':  1, 'T':  0, 'W': -3, 'Y': -2, 'V':  0},
    'R': {'A': -1, 'R':  5, 'N':  0, 'D': -2, 'C': -3, 'Q':  1, 'E':  0, 'G': -2, 'H':  0, 'I': -3, 'L': -2, 'K':  2, 'M': -1, 'F': -3, 'P': -2, 'S': -1, 'T': -1, 'W': -3, 'Y': -2, 'V': -3},
    'N': {'A': -2, 'R':  0, 'N':  6, 'D':  1, 'C': -3, 'Q':  0, 'E':  0, 'G':  0, 'H':  1, 'I': -3, 'L': -3, 'K':  0, 'M': -2, 'F': -3, 'P': -2, 'S':  1, 'T':  0, 'W': -4, 'Y': -2, 'V': -3},
    'D': {'A': -2, 'R': -2, 'N':  1, 'D':  6, 'C': -3, 'Q':  0, 'E':  2, 'G': -1, 'H': -1, 'I': -3, 'L': -4, 'K': -1, 'M': -3, 'F': -3, 'P': -1, 'S':  0, 'T': -1, 'W': -4, 'Y': -3, 'V': -3},
    'C': {'A':  0, 'R': -3, 'N': -3, 'D': -3, 'C':  9, 'Q': -3, 'E': -4, 'G': -3, 'H': -3, 'I': -1, 'L': -1, 'K': -3, 'M': -1, 'F': -2, 'P': -3, 'S': -1, 'T': -1, 'W': -2, 'Y': -2, 'V': -1},
    'Q': {'A': -1, 'R':  1, 'N':  0, 'D':  0, 'C': -3, 'Q':  5, 'E':  2, 'G': -2, 'H':  0, 'I': -3, 'L': -2, 'K':  1, 'M':  0, 'F': -3, 'P': -1, 'S':  0, 'T': -1, 'W': -2, 'Y': -1, 'V': -2},
    'E': {'A': -1, 'R':  0, 'N':  0, 'D':  2, 'C': -4, 'Q':  2, 'E':  5, 'G': -2, 'H':  0, 'I': -3, 'L': -3, 'K':  1, 'M': -2, 'F': -3, 'P': -1, 'S':  0, 'T': -1, 'W': -3, 'Y': -2, 'V': -2},
    'G': {'A':  0, 'R': -2, 'N':  0, 'D': -1, 'C': -3, 'Q': -2, 'E': -2, 'G':  6, 'H': -2, 'I': -4, 'L': -4, 'K': -2, 'M': -3, 'F': -3, 'P': -2, 'S':  0, 'T': -2, 'W': -2, 'Y': -3, 'V': -3},
    'H': {'A': -2, 'R':  0, 'N':  1, 'D': -1, 'C': -3, 'Q':  0, 'E':  0, 'G': -2, 'H':  8, 'I': -3, 'L': -3, 'K': -1, 'M': -2, 'F': -1, 'P': -2, 'S': -1, 'T': -2, 'W': -2, 'Y':  2, 'V': -3},
    'I': {'A': -1, 'R': -3, 'N': -3, 'D': -3, 'C': -1, 'Q': -3, 'E': -3, 'G': -4, 'H': -3, 'I':  4, 'L':  2, 'K': -3, 'M':  1, 'F':  0, 'P': -3, 'S': -2, 'T': -1, 'W': -3, 'Y': -1, 'V':  3},
    'L': {'A': -1, 'R': -2, 'N': -3, 'D': -4, 'C': -1, 'Q': -2, 'E': -3, 'G': -4, 'H': -3, 'I':  2, 'L':  4, 'K': -2, 'M':  2, 'F':  0, 'P': -3, 'S': -2, 'T': -1, 'W': -2, 'Y': -1, 'V':  1},
    'K': {'A': -1, 'R':  2, 'N':  0, 'D': -1, 'C': -3, 'Q':  1, 'E':  1, 'G': -2, 'H': -1, 'I': -3, 'L': -2, 'K':  5, 'M': -1, 'F': -3, 'P': -1, 'S':  0, 'T': -1, 'W': -3, 'Y': -2, 'V': -2},
    'M': {'A': -1, 'R': -1, 'N': -2, 'D': -3, 'C': -1, 'Q':  0, 'E': -2, 'G': -3, 'H': -2, 'I':  1, 'L':  2, 'K': -1, 'M':  5, 'F':  0, 'P': -2, 'S': -1, 'T': -1, 'W': -1, 'Y': -1, 'V':  1},
    'F': {'A': -2, 'R': -3, 'N': -3, 'D': -3, 'C': -2, 'Q': -3, 'E': -3, 'G': -3, 'H': -1, 'I':  0, 'L':  0, 'K': -3, 'M':  0, 'F':  6, 'P': -4, 'S': -2, 'T': -2, 'W':  1, 'Y':  3, 'V': -1},
    'P': {'A': -1, 'R': -2, 'N': -2, 'D': -1, 'C': -3, 'Q': -1, 'E': -1, 'G': -2, 'H': -2, 'I': -3, 'L': -3, 'K': -1, 'M': -2, 'F': -4, 'P':  7, 'S': -1, 'T': -1, 'W': -4, 'Y': -3, 'V': -2},
    'S': {'A':  1, 'R': -1, 'N':  1, 'D':  0, 'C': -1, 'Q':  0, 'E':  0, 'G':  0, 'H': -1, 'I': -2, 'L': -2, 'K':  0, 'M': -1, 'F': -2, 'P': -1, 'S':  4, 'T':  1, 'W': -3, 'Y': -2, 'V': -2},
    'T': {'A':  0, 'R': -1, 'N':  0, 'D': -1, 'C': -1, 'Q': -1, 'E': -1, 'G': -2, 'H': -2, 'I': -1, 'L': -1, 'K': -1, 'M': -1, 'F': -2, 'P': -1, 'S':  1, 'T':  5, 'W': -2, 'Y': -2, 'V':  0},
    'W': {'A': -3, 'R': -3, 'N': -4, 'D': -4, 'C': -2, 'Q': -2, 'E': -3, 'G': -2, 'H': -2, 'I': -3, 'L': -2, 'K': -3, 'M': -1, 'F':  1, 'P': -4, 'S': -3, 'T': -2, 'W': 11, 'Y':  2, 'V': -3},
    'Y': {'A': -2, 'R': -2, 'N': -2, 'D': -3, 'C': -2, 'Q': -1, 'E': -2, 'G': -3, 'H':  2, 'I': -1, 'L': -1, 'K': -2, 'M': -1, 'F':  3, 'P': -3, 'S': -2, 'T': -2, 'W':  2, 'Y':  7, 'V': -1},
    'V': {'A':  0, 'R': -3, 'N': -3, 'D': -3, 'C': -1, 'Q': -2, 'E': -2, 'G': -3, 'H': -3, 'I':  3, 'L':  1, 'K': -2, 'M':  1, 'F': -1, 'P': -2, 'S': -2, 'T':  0, 'W': -3, 'Y': -1, 'V':  4},
}
```

**Tasks:**
1. Write a function `blosum_score(a, b)` that returns the BLOSUM62 score for substituting amino acid `a` with `b`.
2. Modify `global_align` (or write a new function `global_align_blosum`) to use BLOSUM62 instead of simple match/mismatch.
3. Use a gap penalty of -4 (a common choice with BLOSUM62).
4. Align all three hemoglobin pairs (human vs mouse, human vs bovine, mouse vs bovine) using BLOSUM62.
5. Compare the scores to your simple match/mismatch results. Does BLOSUM62 change the alignment?

**Hint:** The BLOSUM62 matrix is defined for the 20 standard amino acids. Your sequences may contain other characters (like `X` for unknown); you can handle these by assigning a score of 0 or -1 for any pair involving an unknown character.

In [ ]:
# BLOSUM62 substitution matrix
# Amino acid order: A R N D C Q E G H I L K M F P S T W Y V
BLOSUM62 = {
    'A': {'A':  4, 'R': -1, 'N': -2, 'D': -2, 'C':  0, 'Q': -1, 'E': -1, 'G':  0, 'H': -2, 'I': -1, 'L': -1, 'K': -1, 'M': -1, 'F': -2, 'P': -1, 'S':  1, 'T':  0, 'W': -3, 'Y': -2, 'V':  0},
    'R': {'A': -1, 'R':  5, 'N':  0, 'D': -2, 'C': -3, 'Q':  1, 'E':  0, 'G': -2, 'H':  0, 'I': -3, 'L': -2, 'K':  2, 'M': -1, 'F': -3, 'P': -2, 'S': -1, 'T': -1, 'W': -3, 'Y': -2, 'V': -3},
    'N': {'A': -2, 'R':  0, 'N':  6, 'D':  1, 'C': -3, 'Q':  0, 'E':  0, 'G':  0, 'H':  1, 'I': -3, 'L': -3, 'K':  0, 'M': -2, 'F': -3, 'P': -2, 'S':  1, 'T':  0, 'W': -4, 'Y': -2, 'V': -3},
    'D': {'A': -2, 'R': -2, 'N':  1, 'D':  6, 'C': -3, 'Q':  0, 'E':  2, 'G': -1, 'H': -1, 'I': -3, 'L': -4, 'K': -1, 'M': -3, 'F': -3, 'P': -1, 'S':  0, 'T': -1, 'W': -4, 'Y': -3, 'V': -3},
    'C': {'A':  0, 'R': -3, 'N': -3, 'D': -3, 'C':  9, 'Q': -3, 'E': -4, 'G': -3, 'H': -3, 'I': -1, 'L': -1, 'K': -3, 'M': -1, 'F': -2, 'P': -3, 'S': -1, 'T': -1, 'W': -2, 'Y': -2, 'V': -1},
    'Q': {'A': -1, 'R':  1, 'N':  0, 'D':  0, 'C': -3, 'Q':  5, 'E':  2, 'G': -2, 'H':  0, 'I': -3, 'L': -2, 'K':  1, 'M':  0, 'F': -3, 'P': -1, 'S':  0, 'T': -1, 'W': -2, 'Y': -1, 'V': -2},
    'E': {'A': -1, 'R':  0, 'N':  0, 'D':  2, 'C': -4, 'Q':  2, 'E':  5, 'G': -2, 'H':  0, 'I': -3, 'L': -3, 'K':  1, 'M': -2, 'F': -3, 'P': -1, 'S':  0, 'T': -1, 'W': -3, 'Y': -2, 'V': -2},
    'G': {'A':  0, 'R': -2, 'N':  0, 'D': -1, 'C': -3, 'Q': -2, 'E': -2, 'G':  6, 'H': -2, 'I': -4, 'L': -4, 'K': -2, 'M': -3, 'F': -3, 'P': -2, 'S':  0, 'T': -2, 'W': -2, 'Y': -3, 'V': -3},
    'H': {'A': -2, 'R':  0, 'N':  1, 'D': -1, 'C': -3, 'Q':  0, 'E':  0, 'G': -2, 'H':  8, 'I': -3, 'L': -3, 'K': -1, 'M': -2, 'F': -1, 'P': -2, 'S': -1, 'T': -2, 'W': -2, 'Y':  2, 'V': -3},
    'I': {'A': -1, 'R': -3, 'N': -3, 'D': -3, 'C': -1, 'Q': -3, 'E': -3, 'G': -4, 'H': -3, 'I':  4, 'L':  2, 'K': -3, 'M':  1, 'F':  0, 'P': -3, 'S': -2, 'T': -1, 'W': -3, 'Y': -1, 'V':  3},
    'L': {'A': -1, 'R': -2, 'N': -3, 'D': -4, 'C': -1, 'Q': -2, 'E': -3, 'G': -4, 'H': -3, 'I':  2, 'L':  4, 'K': -2, 'M':  2, 'F':  0, 'P': -3, 'S': -2, 'T': -1, 'W': -2, 'Y': -1, 'V':  1},
    'K': {'A': -1, 'R':  2, 'N':  0, 'D': -1, 'C': -3, 'Q':  1, 'E':  1, 'G': -2, 'H': -1, 'I': -3, 'L': -2, 'K':  5, 'M': -1, 'F': -3, 'P': -1, 'S':  0, 'T': -1, 'W': -3, 'Y': -2, 'V': -2},
    'M': {'A': -1, 'R': -1, 'N': -2, 'D': -3, 'C': -1, 'Q':  0, 'E': -2, 'G': -3, 'H': -2, 'I':  1, 'L':  2, 'K': -1, 'M':  5, 'F':  0, 'P': -2, 'S': -1, 'T': -1, 'W': -1, 'Y': -1, 'V':  1},
    'F': {'A': -2, 'R': -3, 'N': -3, 'D': -3, 'C': -2, 'Q': -3, 'E': -3, 'G': -3, 'H': -1, 'I':  0, 'L':  0, 'K': -3, 'M':  0, 'F':  6, 'P': -4, 'S': -2, 'T': -2, 'W':  1, 'Y':  3, 'V': -1},
    'P': {'A': -1, 'R': -2, 'N': -2, 'D': -1, 'C': -3, 'Q': -1, 'E': -1, 'G': -2, 'H': -2, 'I': -3, 'L': -3, 'K': -1, 'M': -2, 'F': -4, 'P':  7, 'S': -1, 'T': -1, 'W': -4, 'Y': -3, 'V': -2},
    'S': {'A':  1, 'R': -1, 'N':  1, 'D':  0, 'C': -1, 'Q':  0, 'E':  0, 'G':  0, 'H': -1, 'I': -2, 'L': -2, 'K':  0, 'M': -1, 'F': -2, 'P': -1, 'S':  4, 'T':  1, 'W': -3, 'Y': -2, 'V': -2},
    'T': {'A':  0, 'R': -1, 'N':  0, 'D': -1, 'C': -1, 'Q': -1, 'E': -1, 'G': -2, 'H': -2, 'I': -1, 'L': -1, 'K': -1, 'M': -1, 'F': -2, 'P': -1, 'S':  1, 'T':  5, 'W': -2, 'Y': -2, 'V':  0},
    'W': {'A': -3, 'R': -3, 'N': -4, 'D': -4, 'C': -2, 'Q': -2, 'E': -3, 'G': -2, 'H': -2, 'I': -3, 'L': -2, 'K': -3, 'M': -1, 'F':  1, 'P': -4, 'S': -3, 'T': -2, 'W': 11, 'Y':  2, 'V': -3},
    'Y': {'A': -2, 'R': -2, 'N': -2, 'D': -3, 'C': -2, 'Q': -1, 'E': -2, 'G': -3, 'H':  2, 'I': -1, 'L': -1, 'K': -2, 'M': -1, 'F':  3, 'P': -3, 'S': -2, 'T': -2, 'W':  2, 'Y':  7, 'V': -1},
    'V': {'A':  0, 'R': -3, 'N': -3, 'D': -3, 'C': -1, 'Q': -2, 'E': -2, 'G': -3, 'H': -3, 'I':  3, 'L':  1, 'K': -2, 'M':  1, 'F': -1, 'P': -2, 'S': -2, 'T':  0, 'W': -3, 'Y': -1, 'V':  4},
}


def blosum_score(a, b):
    """
    Return the BLOSUM62 score for substituting amino acid a with b.

    Parameters
    ----------
    a : str — single amino acid character
    b : str — single amino acid character

    Returns
    -------
    int — BLOSUM62 score
    """
    # Your code here
    pass


def global_align_blosum(A, B, gap_penalty=-4):
    """
    Global alignment using the BLOSUM62 substitution matrix.

    Returns
    -------
    score : int — optimal alignment score
    dp : 2-D list — filled DP table
    """
    m, n = len(A), len(B)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    # Your code here — initialize boundaries with gap penalties,
    # then fill using blosum_score
    return dp[m][n], dp


# Your code here — run BLOSUM62 alignment on the hemoglobin pairs

### Exercise 7: Multi-Species BLOSUM62 Alignment (Challenge Extension)

**Stretch goal:** Align **all three** hemoglobin sequences simultaneously.

Since pairwise alignment only compares two sequences at a time, you'll need to:
1. Compute pairwise BLOSUM62 scores for all three pairs.
2. Build a simple distance matrix (e.g., distance = 1 / score, or use the raw score).
3. Discuss: Which species pair is most closely related according to BLOSUM62? Does this match your biological intuition?

In [ ]:
# Your code here — compute pairwise BLOSUM62 scores for all three pairs,
# build a distance matrix, and interpret the results

---
## Further Exploration

### Web Tools for Alignment

- **Freiburg RNA Tools** — interactive Needleman–Wunsch and Smith–Waterman with step-by-step visualization: http://rna.informatik.uni-freiburg.de/Teaching/index.jsp?toolName=Needleman-Wunsch
- **EMBOSS Needle** — command-line and web global alignment: https://www.ebi.ac.uk/Tools/psa/emboss_needle/
- **NCBI BLAST** — heuristic alignment for database searching: https://blast.ncbi.nlm.nih.gov/

### Substitution Matrices

Real protein alignment uses **BLOSUM62** or **PAM** matrices rather than simple match/mismatch scoring. BLOSUM62 is derived from observed frequencies of amino acid substitutions in conserved protein families and is the most commonly used matrix for protein alignment.

### References

- [FASTA format — Wikipedia](https://en.wikipedia.org/wiki/FASTA_format)
- [Needleman–Wunsch algorithm — Wikipedia](https://en.wikipedia.org/wiki/Needleman%E2%80%93Wunsch_algorithm)
- [Smith–Waterman algorithm — Wikipedia](https://en.wikipedia.org/wiki/Smith%E2%80%93Waterman_algorithm)
- [NCBI Datasets](https://www.ncbi.nlm.nih.gov/datasets/)
- [UniProt](https://www.uniprot.org/)
- [Biopython pairwise2 tutorial](https://biopython.org/docs/1.86/Tutorial/chapter_pairwise2.html)